<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "670"
iso_country = "Saint Vincent and the Grenadines"
year = 2021

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    "TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Saint Vincent and the Grenadines': 

Number of trade records present in dataset: 12917
Number of partner countries present in dataset: 132
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$333.9m
Total weight within the dataset: 475.2 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,TandT_match_desc,TandT_ad_val_pct,TandT_N,TandT_ciffob,TandT_cifvalue,TandT_fobvalue,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,670,Saint Vincent and the Grenadines,842,USA,Import,Sea,80111,"080111 | Nuts, edible; coconu",0801,"0801 | Nuts, edible; coconu",08,"08 | Fruit and nuts, edib",131,"Dates, figs, bananas, coconuts, brazil nuts, c...",Agriculture,4,"V_F - Vegetables, fruit, nuts",167.900,0,167.900,kg,589.53,589.53,86.031960,3591.0,2231.343261,191.966834,['Container'],['Container'],3 - Reporter and HS4 code,4.31,1,6.94,167.90,160.96,1. Primary Int. Inv. with Vess Type,1.003378,1.001626,1.001797,1.001797,1.002582,1.002582,1.049584,1.860193,1.157725,1.074840,1.077759,1.104752,1.027727,1.821774,1.218579,1.116200,1.090585,1.139978,1.008558,1.981056,1.511096,1.321551,1.222685,1.188195,0.989743,2.106347,1.733282,1.624232,1.309042,1.273447,0.971284,2.134488,2.039315,2.056849,1.419137,1.337715,0.952455,2.090616,2.003451,2.195075,1.623682,1.362234,0.934716,1.864880,1.943702,2.036796,1.526847,1.378329,0.916628,1.713052,1.741362,2.172124,1.794480,1.387456,131
1,1,670,Saint Vincent and the Grenadines,826,United Kingdom,Import,Sea,80111,"080111 | Nuts, edible; coconu",0801,"0801 | Nuts, edible; coconu",08,"08 | Fruit and nuts, edib",131,"Dates, figs, bananas, coconuts, brazil nuts, c...",Agriculture,4,"V_F - Vegetables, fruit, nuts",156.411,0,156.411,kg,17.41,17.41,80.144996,5566.0,3458.550986,277.185556,['Container'],['Container'],"2 - Reporter, Partner and HS2 code",5.70,5,8.43,156.41,147.98,1. Primary Int. Inv. with Vess Type,1.448802,1.446272,1.446519,1.446519,1.447653,1.447653,1.515520,2.685977,1.671667,1.551987,1.556202,1.595179,1.483960,2.630503,1.759535,1.611708,1.574722,1.646042,1.456281,2.860494,2.181908,1.908220,1.765465,1.715663,1.429114,3.041406,2.502728,2.345268,1.890157,1.838761,1.402460,3.082039,2.944616,2.969934,2.049127,1.931560,1.375272,3.018691,2.892831,3.169521,2.344473,1.966962,1.349659,2.692746,2.806558,2.940980,2.204651,1.990203,1.323542,2.473517,2.514395,3.136383,2.591092,2.003382,131


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [5]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $102.5m (30.7%), ... 

                             cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                            
02 | Meat and edible meat  16563800.8    8068327.1     16.2      4.0
10 | Cereals               14406700.4   51049036.6     14.1     25.2
21 | Miscellaneous edible  13863343.2    3265390.0     13.5      1.6
22 | Beverages, spirits a  12438267.4  101931311.2     12.1     50.3
19 | Preparations of cere   8227117.7    3101050.0      8.0      1.5
04 | Dairy produce; birds   7941908.4    3018659.2      7.7      1.5
16 | Meat, fish, crustace   5377835.7    1203412.0      5.2      0.6
20 | Preparations of vege   4030361.1    2704557.9      3.9      1.3
17 | Sugars and sugar con   3746924.1   10386475.9      3.7      5.1
07 | Vegetables and certa   3605529.5    5603792.4      3.5      2.8
15 | Animal, vegetable or   3172043.2    2238024.0      3.1      1.1
23

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [14]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $103.6m (31.0%), ... 

                    cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                  
Food & Beverages  51582548.6  125593589.8     15.4     26.4
Agriculture       50494443.6   76744422.9     15.1     16.2
Fishing            1478238.3     205739.2      0.4      0.0



### 2.1.3 GTAP (16x headings)

In [7]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $80.2m (24.0%), ... 

                                                cifvalue  vol_kg_vtw  pct_val  \
gtap_desc                                                                       
OFD - Food products n.e.c.                    32694251.1  12546763.8      9.8   
OMT - Meat products n.e.c.                    19447050.4   8649638.2      5.8   
MIL - Dairy products                           8069336.3   3051271.0      2.4   
V_F - Vegetables, fruit, nuts                  4395338.4   6284578.8      1.3   
VOL - Vegetable oils and fats                  3981559.6   4102404.3      1.2   
GRO - Cereal grains n.e.c.                     3650641.2  15088040.9      1.1   
SGR - Sugar                                    3123634.3  10157787.7      0.9   
PDR - Paddy rice                               1460554.0   2536982.4      0.4   
CMT - Bovine meat prods                        1340065.0    383864.3      0.4   
OSD - Oil seeds                             

## 2.2 Comparison of results

In [8]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 30.7% 
GTAP 24.0% 
EORA 31.0%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [9]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [10]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_21395/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,TandT_match_desc,TandT_ad_val_pct,TandT_N,TandT_ciffob,TandT_cifvalue,TandT_fobvalue,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,670,Saint Vincent and the Grenadines,842,USA,Import,Sea,80111,"080111 | Nuts, edible; coconu",0801,"0801 | Nuts, edible; coconu",08,"08 | Fruit and nuts, edib",131,"Dates, figs, bananas, coconuts, brazil nuts, c...",Agriculture,4,"V_F - Vegetables, fruit, nuts",167.900,0,167.900,kg,589.53,589.53,86.031960,3591.0,2231.343261,191.966834,['Container'],['Container'],3 - Reporter and HS4 code,4.31,1,6.94,167.90,160.96,1. Primary Int. Inv. with Vess Type,1.003378,1.001626,1.001797,1.001797,1.002582,1.002582,1.049584,1.860193,1.157725,1.074840,1.077759,1.104752,1.027727,1.821774,1.218579,1.116200,1.090585,1.139978,1.008558,1.981056,1.511096,1.321551,1.222685,1.188195,0.989743,2.106347,1.733282,1.624232,1.309042,1.273447,0.971284,2.134488,2.039315,2.056849,1.419137,1.337715,0.952455,2.090616,2.003451,2.195075,1.623682,1.362234,0.934716,1.864880,1.943702,2.036796,1.526847,1.378329,0.916628,1.713052,1.741362,2.172124,1.794480,1.387456,131,Fruits
1,1,670,Saint Vincent and the Grenadines,826,United Kingdom,Import,Sea,80111,"080111 | Nuts, edible; coconu",0801,"0801 | Nuts, edible; coconu",08,"08 | Fruit and nuts, edib",131,"Dates, figs, bananas, coconuts, brazil nuts, c...",Agriculture,4,"V_F - Vegetables, fruit, nuts",156.411,0,156.411,kg,17.41,17.41,80.144996,5566.0,3458.550986,277.185556,['Container'],['Container'],"2 - Reporter, Partner and HS2 code",5.70,5,8.43,156.41,147.98,1. Primary Int. Inv. with Vess Type,1.448802,1.446272,1.446519,1.446519,1.447653,1.447653,1.515520,2.685977,1.671667,1.551987,1.556202,1.595179,1.483960,2.630503,1.759535,1.611708,1.574722,1.646042,1.456281,2.860494,2.181908,1.908220,1.765465,1.715663,1.429114,3.041406,2.502728,2.345268,1.890157,1.838761,1.402460,3.082039,2.944616,2.969934,2.049127,1.931560,1.375272,3.018691,2.892831,3.169521,2.344473,1.966962,1.349659,2.692746,2.806558,2.940980,2.204651,1.990203,1.323542,2.473517,2.514395,3.136383,2.591092,2.003382,131,Fruits


## 3.3 Inspect the output

In [11]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $73.0m of $333.9m (21.9%)
Volume: 90.3 of 475.2 million kilograms (19.0%)
Value-to-Weight: 0.81 from 0.7 US$ per kg (+15.1%)
Average distance to market (volume-weighted): 2191.7 from 1792.9 nm (+22.2%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    423  25329410.2  11478170.2      7.6      2.4
1        Meat and Prep     44  14054795.5   7445993.2      4.2      1.6
2  Unprocessed Cereals     26  12782629.1  48228024.8      3.8     10.1
3                Dairy    108   7839478.6   2977861.4      2.3      0.6
4           Vegetables    330   4905198.4   6625370.7      1.5      1.4
5      Sugar and Honey     85   3795042.2  10400271.4      1.1      2.2
6        Fats and Oils     85   3113885.2   2169547.8      0.9      0.5
7               Fruits    158   1218478.2    995258.5      0.4      0.2



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [16]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,6282754.5,11058756.9,6909644.6,6424550.7,6440649.5,6605625.3,5821419.7,12676682.0,12110042.2,12233767.3,8453218.3,7974438.5,5499663.5,10199506.5,10369609.2,12872420.5,10629283.9,8269902.6
1,Fats and Oils,2097901.9,3468185.6,2256917.9,2112090.4,2117624.7,2186646.1,1963893.1,3956855.5,3710792.4,3777878.5,2729142.7,2577673.3,1870364.6,3225945.5,3357792.0,4040257.2,3314889.0,2660848.1
2,Fruits,1404204.8,2481322.4,1546906.9,1437087.5,1440859.5,1477300.6,1300161.4,2845972.6,2718947.0,2744222.5,1894588.3,1786490.4,1227562.4,2286550.0,2324478.2,2893456.6,2389905.0,1852817.8
3,Meat and Prep,10697188.3,18957178.4,11798913.6,10954394.6,10984115.9,11259298.8,9899319.1,21752250.1,20782323.7,20961423.2,14462737.6,13633080.1,9342391.6,17457995.5,17746538.0,22135209.1,18286681.3,14139986.2
4,Processed Cereals,10753525.7,18731437.9,11774604.2,10972288.2,10996628.4,11287398.8,9982892.3,21451067.6,20488803.4,20747922.9,14368352.4,13564316.9,9446050.6,17321747.3,17612373.7,21704619.6,17914461.3,14063194.4
5,Sugar and Honey,8323153.7,13478991.3,8883361.5,8342583.7,8361411.7,8649349.4,7817873.9,15341338.3,14347589.5,14669648.9,10689058.9,10110445.0,7465815.2,12590593.8,13147279.2,15607710.0,12783215.9,10427933.1
6,Unprocessed Cereals,66964353.6,113796357.0,72559016.4,67976695.1,68076690.7,70013751.6,62440720.3,129837798.9,123965257.3,126281833.3,87918331.8,83232393.2,59299871.8,105824943.7,107652118.7,130303128.0,107355274.7,86254151.8
7,Vegetables,12721676.5,22482244.6,14014534.8,13019430.0,13053794.2,13383920.8,11778983.5,25791360.3,24639149.3,24868344.2,17165906.6,16186025.1,11120811.1,20717820.2,21062105.4,26215286.0,21653501.6,16786642.3


### 3.3.2 Total FEM costs by scenario (% cif)

In [15]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.61,0.08,0.02,0.02,0.04,0.87,0.80,0.82,0.34,0.27,0.60,0.62,0.94,0.65,0.35
1,Fats and Oils,0.44,0.05,0.00,0.01,0.03,0.64,0.56,0.58,0.25,0.20,0.44,0.48,0.70,0.46,0.25
2,Fruits,0.88,0.12,0.03,0.03,0.06,1.27,1.16,1.19,0.49,0.40,0.87,0.90,1.37,0.95,0.51
3,Meat and Prep,0.59,0.08,0.02,0.02,0.04,0.84,0.77,0.79,0.32,0.27,0.58,0.60,0.91,0.64,0.34
4,Processed Cereals,0.31,0.04,0.01,0.01,0.02,0.45,0.41,0.43,0.17,0.14,0.31,0.32,0.48,0.33,0.18
5,Sugar and Honey,1.36,0.15,0.01,0.01,0.09,1.98,1.72,1.81,0.76,0.60,1.35,1.50,2.15,1.40,0.78
6,Unprocessed Cereals,3.66,0.44,0.08,0.09,0.24,5.27,4.81,4.99,1.99,1.63,3.64,3.78,5.55,3.76,2.11
7,Vegetables,1.99,0.26,0.06,0.07,0.14,2.86,2.62,2.67,1.10,0.90,1.96,2.03,3.08,2.15,1.16
